#### Install and Import all required libraries

In [1]:
# Mount Google Drive and import all libraries
import os
import sys
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

!pip install torch-geometric torch-cluster pymatgen

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader, Batch
from torch_geometric.nn import CGConv, global_mean_pool, GATConv, Set2Set, MessagePassing
from torch_geometric.explain import GNNExplainer
from torch_cluster import radius_graph
from torch_geometric.utils import scatter
from pymatgen.io.cif import CifParser
from pymatgen.core import Structure, Element
from torch.utils.data import Subset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

import matplotlib.pyplot as plt
import seaborn as sns

!pip install pysr
from pysr import PySRRegressor

import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility (critical for scientific research)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define paths
BASE_DIR = "/content/drive/MyDrive/IML Project"

DATA_PATH = os.path.join(BASE_DIR, "Dataset.csv")
CIF_DIR = os.path.join(BASE_DIR)

GRAPH_DIR = os.path.join(BASE_DIR, "graphs")
os.makedirs(GRAPH_DIR, exist_ok=True)
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1

/usr/local/lib/python3.12/dist-packages/juliacall/__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(


[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliapkg/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Using Julia 1.11.5 at /usr/local/bin/julia
[juliapkg] Using Julia project at /root/.julia/environments/pyjuliapkg
[juliapkg] Writing Project.toml:
           | [deps]
           | SymbolicRegression = "8254be44-1295-4e6a-a16d-46603ac705cb"
           | Serialization = "9e88b42a-f829-5b0c-bbe9-9e923198166b"
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
           | OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b2a3a95"
           | 
           | [compat]
           | SymbolicRegression = "~1.11"
           | Serialization = "^1"
           | PythonCall = "=0.9.26"
           | OpenSSL_jll = "~3.0"
[juliapkg] Installing packages:
           | impo

 Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Total samples:", len(df))

Prepare Tabular Data

In [ ]:
# Check class balance
print("\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
# Build absolute CIF paths
df['cif_path'] = df['cif'].apply(lambda x: os.path.join(CIF_DIR, x))

# Quick check
print(df[['cif', 'cif_path']].head())

# Train / Val / Test Split (STRATIFIED)

In [ ]:
# SPLIT DATA
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=SEED
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

#   Prepare graphical data

# Feature Extraction

In [ ]:
# FEATURE EXTRACTION
def get_atom_features(element):
    el = Element(element)

    return [
        el.Z,
        el.X if el.X else 0,
        el.atomic_radius if el.atomic_radius else 0,
        el.group if el.group else 0,
        el.row if el.row else 0
    ]


def extract_global_features(row, structure):
    return [
        row['spacegroup_number'],
        row['number of atoms'],
        row['a'], row['b'], row['c'],
        row['alpha'], row['beta'], row['gamma'],
        row['Z'],
        row['electronegativity'],
        row['Band Gap'],
        structure.volume,
        structure.density,
        len(structure.composition.elements)
    ]

# Collect TRAIN Features (for scaling ONLY)

In [ ]:
# COLLECT TRAIN FEATURES
node_samples = []
edge_samples = []
global_samples = []

cutoff = 6.0

for _, row in train_df.iterrows():
    try:
        structure = Structure.from_file(row['cif_path'])

        # Node features
        for site in structure:
            node_samples.append(get_atom_features(site.specie.symbol))

        # Edge features (distance)
        for i, site in enumerate(structure):
            neighbors = structure.get_neighbors(site, cutoff)
            for n in neighbors:
                edge_samples.append([n.nn_distance])

        # Global features
        global_samples.append(extract_global_features(row, structure))

    except Exception as e:
        print(f"Error processing {row['cif_path']} | {e}")

# Convert safely
node_samples = np.array(node_samples)
edge_samples = np.array(edge_samples)
global_samples = np.array(global_samples)

print("Node shape:", node_samples.shape)
print("Edge shape:", edge_samples.shape)
print("Global shape:", global_samples.shape)

Node shape: (211448, 5)
Edge shape: (11843614, 1)
Global shape: (6767, 14)


# Fit Scalers (TRAIN ONLY)

In [ ]:
# SCALING

node_scaler = StandardScaler().fit(node_samples)
edge_scaler = StandardScaler().fit(edge_samples)
global_scaler = StandardScaler().fit(global_samples)

# Gaussian Expansion

In [ ]:
# GAUSSIAN EXPANSION

class GaussianDistance:
    def __init__(self, dmin=0, dmax=6, step=0.2):
        self.filter = np.arange(dmin, dmax + step, step)
        self.var = step

    def expand(self, distances):
        return np.exp(-(distances[..., np.newaxis] - self.filter)**2 / self.var**2)

gaussian_expansion = GaussianDistance()

# Building Graphs

In [ ]:
def build_graph(row):
    structure = Structure.from_file(row['cif_path'])

    # NODE FEATURES
    node_feat = [get_atom_features(site.specie.symbol) for site in structure]
    node_feat = node_scaler.transform(node_feat)
    x = torch.tensor(node_feat, dtype=torch.float)

    # POSITIONS
    pos = torch.tensor(structure.cart_coords, dtype=torch.float)

    # EDGES
    edge_index = []
    edge_attr = []

    cutoff = 6.0

    for i, site in enumerate(structure):
        neighbors = structure.get_neighbors(site, cutoff)

        for n in neighbors:
            j = n.index
            dist = n.nn_distance

            edge_index.append([i, j])
            edge_attr.append([dist])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    edge_attr = np.array(edge_attr)
    edge_attr = edge_scaler.transform(edge_attr)
    edge_attr = gaussian_expansion.expand(edge_attr.squeeze())
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    # GLOBAL FEATURES
    global_feat = extract_global_features(row, structure)
    global_feat = global_scaler.transform([global_feat])
    u = torch.tensor(global_feat, dtype=torch.float)

    # LABEL
    y = torch.tensor([row['label']], dtype=torch.float)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y,
        pos=pos
    )

    data.u = u

    return data

In [ ]:
# BUILD & SAVE GRAPHS

print("Building graphs")

for _, row in tqdm(df.iterrows(), total=len(df)):
    graph_path = os.path.join(GRAPH_DIR, f"{row['mp_material_id']}.pt")

    if os.path.exists(graph_path):
        continue

    try:
        graph = build_graph(row)
        torch.save(graph, graph_path)
    except Exception as e:
        print(f"Error: {row['mp_material_id']} | {e}")

Building graphs


100%|██████████| 9668/9668 [50:10<00:00,  3.21it/s]


# Model 1. CharlesCGCNN

In [2]:

class CharlesCGCNN(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dim=128, dropout=0.2):
        super(CharlesCGCNN, self).__init__()

        # Embeddings
        self.node_emb = nn.Linear(node_dim, hidden_dim)
        self.global_emb = nn.Linear(global_dim, hidden_dim)

        # CGCNN Layers
        self.conv1 = CGConv(hidden_dim, dim=edge_dim)
        self.conv2 = CGConv(hidden_dim, dim=edge_dim)
        self.conv3 = CGConv(hidden_dim, dim=edge_dim)

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Fully Connected Layers
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        batch = data.batch
        u = data.u  # global features

        # Embedding
        x = self.node_emb(x)
        u = self.global_emb(u)

        # CGCNN Layers (with gating)
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index, edge_attr)
        x = F.relu(x)

        # Pooling
        x = global_mean_pool(x, batch)

        # Combine with global features
        combined = torch.cat([x, u], dim=1)

        # Final prediction
        out = self.fc(combined)

        return out

MODEL INITIALIZATION

In [3]:
model = CharlesCGCNN(
    node_dim=5,          # from atom features
    edge_dim=31,         # from Gaussian expansion
    global_dim=14
).to(device)

In [4]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

# Dataset + DataLoaders

In [5]:
# LOAD SAVED GRAPHS

class CrystalDataset(Dataset):
    def __init__(self, graph_dir):
        super().__init__()
        self.graph_dir = graph_dir

        # Ensure consistent ordering
        self.files = sorted([
            f for f in os.listdir(graph_dir)
            if f.endswith(".pt")
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.graph_dir, self.files[idx])

        # weights_only=False
        data = torch.load(path, map_location="cpu", weights_only=False)

        return data

In [6]:
# CREATE DATASET

dataset = CrystalDataset(GRAPH_DIR)

print("Total graphs:", len(dataset))

Total graphs: 9668


In [7]:
# EXTRACT LABELS
labels = []
valid_indices = []

for i in range(len(dataset)):
    try:
        data = dataset[i]
        labels.append(data.y.item())
        valid_indices.append(i)
    except Exception as e:
        print(f"Skipping {i}: {e}")

print("Valid samples:", len(valid_indices))


Valid samples: 9668


In [8]:
# STRATIFIED SPLIT
train_idx, temp_idx = train_test_split(
    valid_indices,
    test_size=0.30,
    stratify=labels,
    random_state=SEED
)


In [9]:
# Align labels for second split
temp_labels = [labels[valid_indices.index(i)] for i in temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=temp_labels,
    random_state=SEED
)


In [10]:
# CREATE SUBSETS
train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)



In [11]:
# DATALOADERS
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)


In [12]:
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 6767 | Val: 1450 | Test: 1451


# Training Loop

In [ ]:

def train_epoch(loader):
    model.train()
    total_loss = 0

    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)
        loss = criterion(out, data.y.unsqueeze(1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def eval_epoch(loader):
    model.eval()
    total_loss = 0
    preds, labels = [], []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            out = model(data)
            loss = criterion(out, data.y.unsqueeze(1))

            total_loss += loss.item()

            probs = torch.sigmoid(out)
            preds.append(probs.cpu())
            labels.append(data.y.cpu())

    return total_loss / len(loader), torch.cat(preds), torch.cat(labels)


# Training control
best_val_loss = float('inf')
patience = 15
counter = 0

EPOCHS = 100

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(train_loader)
    val_loss, _, _ = eval_epoch(val_loader)

    scheduler.step(val_loss)

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

Epoch 001 | Train Loss: 27.0314 | Val Loss: 0.6630
Epoch 002 | Train Loss: 1.2741 | Val Loss: 0.5456
Epoch 003 | Train Loss: 0.5288 | Val Loss: 0.4889
Epoch 004 | Train Loss: 0.5034 | Val Loss: 0.4795
Epoch 005 | Train Loss: 0.5028 | Val Loss: 0.4777
Epoch 006 | Train Loss: 0.4836 | Val Loss: 0.4740
Epoch 007 | Train Loss: 0.4726 | Val Loss: 0.4611
Epoch 008 | Train Loss: 0.4632 | Val Loss: 0.4599
Epoch 009 | Train Loss: 0.4944 | Val Loss: 0.4887
Epoch 010 | Train Loss: 0.4553 | Val Loss: 0.4484
Epoch 011 | Train Loss: 0.4546 | Val Loss: 0.4609
Epoch 012 | Train Loss: 0.4498 | Val Loss: 0.4680
Epoch 013 | Train Loss: 0.4474 | Val Loss: 0.4634
Epoch 014 | Train Loss: 0.4364 | Val Loss: 0.4417
Epoch 015 | Train Loss: 0.4485 | Val Loss: 0.4769
Epoch 016 | Train Loss: 0.4434 | Val Loss: 0.4511
Epoch 017 | Train Loss: 0.4394 | Val Loss: 0.4519
Epoch 018 | Train Loss: 0.4311 | Val Loss: 0.4461
Epoch 019 | Train Loss: 0.4784 | Val Loss: 0.4430
Epoch 020 | Train Loss: 0.4418 | Val Loss: 0.4451

# Evaluation

In [ ]:

def evaluate_model(loader, name="Test"):
    model.eval()
    preds, labels_list = [], []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            out = model(data)
            probs = torch.sigmoid(out)

            preds.append(probs.cpu())
            labels_list.append(data.y.cpu())

    probs = torch.cat(preds).numpy()
    labels = torch.cat(labels_list).numpy()
    binary = (probs > 0.5).astype(int)

    print(f"\n{name} Performance:")
    print(f"Accuracy : {accuracy_score(labels, binary):.4f}")
    print(f"F1-score : {f1_score(labels, binary):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(labels, probs):.4f}")
    print(classification_report(labels, binary))

    return labels, probs, binary


# Load best model
model.load_state_dict(torch.load("best_model.pt"))

labels, probs, binary = evaluate_model(test_loader)

# PLOT TRAINING CURVES

In [ ]:
# PLOT TRAINING CURVES

epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 5))

In [ ]:
# LOSS CURVES
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

In [ ]:

# METRICS
plt.subplot(1, 2, 2)
plt.plot(epochs_range, val_f1_scores, label="Val F1")
plt.plot(epochs_range, val_roc_scores, label="Val ROC-AUC")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation Metrics")
plt.legend()

plt.tight_layout()
plt.show()

# Confusion Matrix + ROC

In [ ]:
# VISUALIZATION
cm = confusion_matrix(labels, binary)
ConfusionMatrixDisplay(cm).plot()
plt.title("Confusion Matrix")
plt.show()

RocCurveDisplay.from_predictions(labels, probs)
plt.title("ROC Curve")
plt.show()

#Scatter Plot

In [ ]:
import seaborn as sns

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))

sns.stripplot(
    x=test_labels,
    y=test_probs,
    hue=test_labels,
    palette=['steelblue', 'coral'],
    jitter=0.15,
    alpha=0.4,
    legend=False
)

plt.axhline(0.5, linestyle='--', linewidth=1)

plt.xticks([0, 1], ['Trivial (0)', 'Topological (1)'])
plt.xlabel('True Label')
plt.ylabel('Predicted P(TI)')
plt.title('Predicted Probability vs True Label')

plt.tight_layout()
plt.show()

# Model 2.  Graph Attention Networks(GAT)

In [ ]:
class AttentionGNN(nn.Module):
    def __init__(self, node_dim, global_dim):
        super().__init__()

        self.conv1 = GATConv(node_dim, 64, heads=4, concat=True, dropout=0.2)
        self.conv2 = GATConv(64 * 4, 64, dropout=0.2)

        self.norm1 = nn.BatchNorm1d(64 * 4)
        self.norm2 = nn.BatchNorm1d(64)

        self.global_emb = nn.Sequential(
            nn.Linear(global_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, data, return_attention=False):
        x, edge_index, batch, u = data.x, data.edge_index, data.batch, data.u

        # GAT layer 1
        if return_attention:
            x, (edge_index1, attn1) = self.conv1(
                x, edge_index, return_attention_weights=True
            )
        else:
            x = self.conv1(x, edge_index)

        x = self.norm1(x)
        x = F.relu(x)

        # GAT layer 2
        if return_attention:
            x, (edge_index2, attn2) = self.conv2(
                x, edge_index, return_attention_weights=True
            )
        else:
            x = self.conv2(x, edge_index)

        x = self.norm2(x)
        x = F.relu(x)

        x = global_mean_pool(x, batch)
        u = self.global_emb(u)

        out = self.fc(torch.cat([x, u], dim=1))

        if return_attention:
            return out, (edge_index1, attn1, edge_index2, attn2)

        return out

In [ ]:
criterion = nn.BCEWithLogitsLoss()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

In [ ]:
def train(model, loader):
    model.train()
    total_loss = 0

    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()

        out = model(data).view(-1)
        y = data.y.float()

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate(model, loader):
    model.eval()

    preds = []
    labels = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            out = model(data).view(-1)
            prob = torch.sigmoid(out)

            preds.extend(prob.cpu().numpy())
            labels.extend(data.y.cpu().numpy())

    preds_bin = (np.array(preds) > 0.5).astype(int)

    acc = accuracy_score(labels, preds_bin)
    f1  = f1_score(labels, preds_bin)
    auc = roc_auc_score(labels, preds)

    return acc, f1, auc

In [ ]:
model = AttentionGNN(node_dim, global_dim).to(device)

for epoch in range(1, 51):
    loss = train(model, train_loader)
    acc, f1, auc = evaluate(model, val_loader)

    print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

In [ ]:
model.eval()

data_sample = next(iter(test_loader)).to(device)

out, attn_data = model(data_sample, return_attention=True)

edge_index1, attn1, edge_index2, attn2 = attn_data

attn1 = attn1.detach().cpu().numpy()
attn2 = attn2.detach().cpu().numpy()

# ATTENTION HEATMAP

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 3))
sns.heatmap(attn1.T, cmap="viridis", cbar=True)

plt.title("GAT Layer 1 Attention Weights")
plt.xlabel("Edges")
plt.ylabel("Heads")
plt.show()

# GRAPH VISUALIZATION WITH ATTENTION

In [ ]:
import networkx as nx
import numpy as np

edge_index = edge_index1.cpu().numpy()

# Average over heads
edge_weights = attn1.mean(axis=1)

G = nx.Graph()

for i in range(edge_index.shape[1]):
    u, v = edge_index[:, i]
    G.add_edge(u, v, weight=edge_weights[i])

pos = nx.spring_layout(G)

weights = [G[u][v]['weight'] for u, v in G.edges()]

plt.figure(figsize=(6,6))
nx.draw(
    G,
    pos,
    node_size=50,
    edge_color=weights,
    edge_cmap=plt.cm.plasma,
    width=2
)

plt.title("Attention-based Edge Importance")
plt.show()

# GNNExplainer (Interpretability)

In [ ]:
# GNN EXPLAINER

explainer = GNNExplainer(model, epochs=200)

node_mask, edge_mask = explainer.explain_graph(
    x=data_sample.x,
    edge_index=data_sample.edge_index
)


# NODE FEATURE HEATMAPS

In [ ]:
plt.figure(figsize=(10, 2))
sns.heatmap(node_mask.reshape(1, -1), cmap="viridis", cbar=True)

plt.title("Node Feature Importance (GNNExplainer)")
plt.yticks([])
plt.xlabel("Feature Index")
plt.show()

# EDGE IMPORTANCE HEATMAPS

In [ ]:
plt.figure(figsize=(10, 2))
sns.heatmap(edge_mask.reshape(1, -1), cmap="inferno", cbar=True)

plt.title("Edge Importance (GNNExplainer)")
plt.yticks([])
plt.xlabel("Edge Index")
plt.show()

In [ ]:
node_mask = node_mask.detach().cpu().numpy()
edge_mask = edge_mask.detach().cpu().numpy()

node_mask = (node_mask - node_mask.min()) / (node_mask.max() - node_mask.min() + 1e-8)
edge_mask = (edge_mask - edge_mask.min()) / (edge_mask.max() - edge_mask.min() + 1e-8)

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(edge_mask.reshape(1, -1), cmap="inferno", cbar=True)

plt.title("Edge Importance")
plt.xlabel("Edge Index")
plt.yticks([])
plt.show()

# Attention Weights (from GAT)

In [ ]:
import networkx as nx

edge_index = data_sample.edge_index.cpu().numpy()

G = nx.Graph()

for i in range(edge_index.shape[1]):
    u, v = edge_index[:, i]
    G.add_edge(u, v, weight=edge_mask[i])

pos = nx.spring_layout(G)

edges = G.edges()
weights = [G[u][v]['weight'] for u, v in edges]

plt.figure(figsize=(6,6))
nx.draw(
    G,
    pos,
    with_labels=True,
    node_color=node_mask.mean(axis=1) if node_mask.ndim > 1 else node_mask,
    edge_color=weights,
    edge_cmap=plt.cm.plasma,
    width=2
)
plt.title("Graph Explanation (Edge Importance)")
plt.show()

In [ ]:
# SYMBOLIC REGRESSION
# Use global features as input
X = global_samples  # from earlier
y = df['label'].values

model_sr = PySRRegressor(
    niterations=40,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["sqrt", "log", "exp"],
)

model_sr.fit(X, y)

print(model_sr)